# Issue #11 [Data/KPI]: Capa Analítica Split–Apply–Combine y Fichas Técnicas de KPI para YEB

**Subentorno:** `analytics/` | **Rama:** `feat/JMCM-analytics-nosql` | **Script equivalente:** `scripts/11_kpi_agregaciones_yeb.py`  
**Criterios de Aceptación:** `CA-ISS-11-01` a `CA-ISS-11-06` | **Riesgos mitigados:** `NEG-009`, `TEC-016`, `LEG-001` | **Casos de prueba:** `CP-051`, `CP-051B`, `CP-051C`

Este cuaderno consume de forma inmutable `data/processed/yeb_postulaciones_transformado.csv` (base enriquecida en la Semana 5) y aplica el patrón *Split–Apply–Combine* (`groupby()`, `agg()`, `value_counts()`, `crosstab()`, `pivot_table()`) para resumir el comportamiento de las postulaciones por ciudad, modalidad y grupo etario (menores protegidos de 16–17 años frente a adultos jóvenes de 18–24).

## 1. Universo analizable y diferenciación `size()`, `count()` y `nunique()` (`CA-ISS-11-01` / `NEG-009` / `TEC-016`)

- **Total de filas recibidas (`97`):** incluye los `93` registros limpios (`origen_registro == 'dataset_preparado'`) y los `4` registros aislados en `cuarentena_issue_05` (`P1 - Revisar identidad o edad`).
- **Universo analizable (`93`):** subconjunto válido `origen_registro == 'dataset_preparado'` al corte `2026-02-02`.
- **Regla de denominadores:**
  - `size()`: total de postulaciones (filas) del grupo, incluyendo nulos en columnas condicionales.
  - `count()`: valores no nulos de la variable (`55` postulaciones respondidas en `tiempo_respuesta_dias`; `38` activas en `dias_sin_actualizar`).
  - `nunique()`: entidades distintas sin duplicar (`93` candidatos únicos y `18` vacantes únicas en el universo analizable).

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd
elif (cwd / 'analytics' / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd / 'analytics'
else:
    ANALYTICS_ROOT = cwd.parent

RUTA_FUENTE = ANALYTICS_ROOT / 'data' / 'processed' / 'yeb_postulaciones_transformado.csv'
HUELLA_ANTES = hashlib.sha256(RUTA_FUENTE.read_bytes()).hexdigest()

FECHAS = ['job_published_at', 'applied_at', 'updated_at']
df_total = pd.read_csv(RUTA_FUENTE, parse_dates=FECHAS)
for col in ('is_protected_minor', 'minor_permit_verified'):
    df_total[col] = df_total[col].astype(str).str.strip().str.lower().eq('true')
df_total['grupo_etario'] = df_total['is_protected_minor'].map({True: 'Menor protegido (16-17)', False: 'Joven adulto (18-24)'})

df_u = df_total.loc[df_total['origen_registro'] == 'dataset_preparado'].copy()
FECHA_CORTE = df_u['updated_at'].max()

comparativa_denominadores = pd.DataFrame([
    {'Conjunto': 'Recibido total (incluye cuarentena P1)', 'size() filas': df_total['application_id'].size, 'count(tiempo_respuesta_dias)': int(df_total['tiempo_respuesta_dias'].count()), 'nunique(application_id)': df_total['application_id'].nunique(), 'nunique(documento_hash)': df_total['documento_hash'].nunique(), 'nunique(candidate_hash)': df_total['candidate_hash'].nunique(), 'nunique(job_id)': df_total['job_id'].nunique()},
    {'Conjunto': 'Universo analizable (dataset_preparado)', 'size() filas': df_u['application_id'].size, 'count(tiempo_respuesta_dias)': int(df_u['tiempo_respuesta_dias'].count()), 'nunique(application_id)': df_u['application_id'].nunique(), 'nunique(documento_hash)': df_u['documento_hash'].nunique(), 'nunique(candidate_hash)': df_u['candidate_hash'].nunique(), 'nunique(job_id)': df_u['job_id'].nunique()},
])
assert len(df_total) == 97 and len(df_u) == 93
assert df_u['tiempo_respuesta_dias'].count() == 55 and df_u['dias_sin_actualizar'].count() == 38
print(f'Fecha de corte: {FECHA_CORTE.date()} | Huella SHA-256 fuente: {HUELLA_ANTES[:16]}...')
comparativa_denominadores

Fecha de corte: 2026-02-02 | Huella SHA-256 fuente: 769e94c0a85a651f...


,Conjunto,size() filas,count(tiempo_respuesta_dias),nunique(application_id),nunique(documento_hash),nunique(candidate_hash),nunique(job_id)
0,Recibido total (incluye cuarentena P1),97,56,96,96,97,18
1,Universo analizable (dataset_preparado),93,55,93,93,93,18


## 2. Agregaciones `groupby().agg()` conservando `NaN` en grupos con `count <= 1` (`CA-ISS-11-02` / `CP-051`)

La desviación estándar muestral (`std`, `ddof=1`) divide por `count - 1`. Cuando un subgrupo solo tiene una observación (`count == 1`), la dispersión muestral no está definida y Pandas devuelve `NaN`. Reemplazar ese `NaN` por `0` afirmaría falsamente que todas las observaciones del grupo fueron idénticas.

In [2]:
def resumir_segmento(df: pd.DataFrame, cols_grupo: list[str], dimension: str) -> pd.DataFrame:
    res = df.groupby(cols_grupo, dropna=False, observed=False).agg(
        n_filas_size=('application_id', 'size'),
        n_candidatos_nunique=('candidate_hash', 'nunique'),
        n_vacantes_nunique=('job_id', 'nunique'),
        n_respondidas_count=('tiempo_respuesta_dias', 'count'),
        tiempo_resp_media=('tiempo_respuesta_dias', 'mean'),
        tiempo_resp_mediana=('tiempo_respuesta_dias', 'median'),
        tiempo_resp_std=('tiempo_respuesta_dias', 'std'),
        n_salarios_count=('expected_salary_cop', 'count'),
        salario_esp_media=('expected_salary_cop', 'mean'),
        salario_esp_mediana=('expected_salary_cop', 'median'),
        salario_esp_std=('expected_salary_cop', 'std'),
        brecha_salarial_media_pct=('brecha_salarial_aspiracion_pct', 'mean'),
        brecha_salarial_mediana_pct=('brecha_salarial_aspiracion_pct', 'median'),
        brecha_salarial_std_pct=('brecha_salarial_aspiracion_pct', 'std'),
        n_entrevista_o_contratado=('status', lambda s: int(s.isin(['interview', 'accepted']).sum())),
        tasa_entrevista_pct=('status', lambda s: round(100.0 * float(s.isin(['interview', 'accepted']).mean()), 2)),
        n_contratados=('status', lambda s: int((s == 'accepted').sum())),
        tasa_contratacion_pct=('status', lambda s: round(100.0 * float((s == 'accepted').mean()), 2)),
    )
    res = res[res['n_filas_size'] > 0].reset_index()
    assert res.loc[res['n_respondidas_count'] <= 1, 'tiempo_resp_std'].isna().all()
    assert res.loc[res['n_salarios_count'] <= 1, 'salario_esp_std'].isna().all()
    res.insert(0, 'dimension', dimension)
    res.insert(1, 'segmento', res[cols_grupo].astype(str).agg(' | '.join, axis=1))
    return res

tabla_ciudad_mod = resumir_segmento(df_u, ['ciudad_normalizada', 'modality'], 'Ciudad x Modalidad')
tabla_ciudad_mod[['segmento', 'n_filas_size', 'n_respondidas_count', 'tiempo_resp_media', 'tiempo_resp_mediana', 'tiempo_resp_std', 'salario_esp_mediana', 'salario_esp_std', 'tasa_entrevista_pct']].round(2)

,segmento,n_filas_size,n_respondidas_count,tiempo_resp_media,tiempo_resp_mediana,tiempo_resp_std,salario_esp_mediana,salario_esp_std,tasa_entrevista_pct
0,Barranquilla | Híbrido,1,1,7.00,7.0,NaN,1305884.0,NaN,100.00
1,Barranquilla | Presencial,14,9,4.00,4.0,2.65,2133032.5,458658.30,35.71
2,Bogotá | Híbrido,16,11,4.55,4.0,2.42,2036990.5,385894.69,43.75
3,Bogotá | Remoto,1,0,NaN,NaN,NaN,1387625.0,NaN,0.00
4,Bucaramanga | Híbrido,14,7,3.71,3.0,2.14,2233809.5,373928.96,28.57
5,Cali | Presencial,1,1,0.00,0.0,NaN,1744353.0,NaN,100.00
6,Cali | Remoto,15,9,3.11,4.0,2.67,2024196.0,298853.42,46.67
7,Medellín | Presencial,15,8,4.00,3.0,2.67,1843002.0,350233.87,40.00
8,Pereira | Remoto,16,9,4.56,5.0,2.35,1843937.5,404966.65,37.50


In [3]:
tabla_ciudad_mod_edad = resumir_segmento(df_u, ['ciudad_normalizada', 'modality', 'grupo_etario'], 'Ciudad x Modalidad x Grupo etario')
tabla_ciudad_mod_edad[['segmento', 'n_filas_size', 'n_respondidas_count', 'tiempo_resp_media', 'tiempo_resp_std', 'salario_esp_mediana', 'salario_esp_std', 'tasa_entrevista_pct']].round(2)

,segmento,n_filas_size,n_respondidas_count,tiempo_resp_media,tiempo_resp_std,salario_esp_mediana,salario_esp_std,tasa_entrevista_pct
0,Barranquilla | Híbrido | Joven adulto (18-24),1,1,7.00,NaN,1305884.0,NaN,100.00
1,Barranquilla | Presencial | Joven adulto (18-24),9,4,4.50,2.65,2177242.0,543352.37,33.33
2,Barranquilla | Presencial | Menor protegido (1...,5,5,3.60,2.88,2088823.0,301798.50,40.00
3,Bogotá | Híbrido | Joven adulto (18-24),13,10,4.30,2.41,1972608.0,352608.85,46.15
4,Bogotá | Híbrido | Menor protegido (16-17),3,1,7.00,NaN,2189963.0,596171.52,33.33
5,Bogotá | Remoto | Joven adulto (18-24),1,0,NaN,NaN,1387625.0,NaN,0.00
6,Bucaramanga | Híbrido | Joven adulto (18-24),10,4,3.00,2.45,2271593.5,387756.00,30.00
7,Bucaramanga | Híbrido | Menor protegido (16-17),4,3,4.67,1.53,2211745.0,391395.00,25.00
8,Cali | Presencial | Joven adulto (18-24),1,1,0.00,NaN,1744353.0,NaN,100.00
9,Cali | Remoto | Joven adulto (18-24),13,9,3.11,2.67,2024196.0,313523.89,53.85


## 3. Tablas de participación `value_counts(normalize=True, dropna=False)` y tablas cruzadas `crosstab(..., normalize='index', margins=True)` (`CA-ISS-11-03` / `CP-051B`)

In [4]:
part_modalidad = (df_u['modality'].value_counts(normalize=True, dropna=False) * 100).round(2).rename('participacion_%').to_frame()
part_tiempo_con_nan = (df_u['tiempo_respuesta_dias'].value_counts(normalize=True, dropna=False) * 100).round(2).rename('participacion_%').to_frame()
assert np.isclose(part_modalidad['participacion_%'].sum(), 100.0, atol=0.05)
assert part_tiempo_con_nan.index.isna().any(), 'dropna=False debe conservar el bloque NaN'
print('Participación por modalidad (%):')
display(part_modalidad)
print('\nParticipación de tiempo_respuesta_dias con dropna=False (visibilizando el 40,86% de activas en NaN):')
display(part_tiempo_con_nan.head(6))

Participación por modalidad (%):


,participacion_%
modality,
Remoto,34.41
Híbrido,33.33
Presencial,32.26



Participación de tiempo_respuesta_dias con dropna=False (visibilizando el 40,86% de activas en NaN):


,participacion_%
tiempo_respuesta_dias,
NaN,40.86
3.0,10.75
6.0,9.68
1.0,6.45
7.0,6.45
2.0,5.38


In [5]:
ct_mod_estado_pct = (pd.crosstab(df_u['modality'], df_u['status'], normalize='index', margins=True) * 100).round(2)
ct_edad_estado_pct = (pd.crosstab(df_u['grupo_etario'], df_u['status'], normalize='index', margins=True) * 100).round(2)
assert np.allclose(pd.crosstab(df_u['modality'], df_u['status'], normalize='index', margins=True).sum(axis=1), 1.0)
assert np.allclose(pd.crosstab(df_u['grupo_etario'], df_u['status'], normalize='index', margins=True).sum(axis=1), 1.0)
print('Tabla cruzada: Modalidad vs. Estado de postulación (% por fila, margins=True):')
display(ct_mod_estado_pct)
print('\nTabla cruzada: Grupo etario vs. Estado de postulación (% por fila, margins=True):')
display(ct_edad_estado_pct)

Tabla cruzada: Modalidad vs. Estado de postulación (% por fila, margins=True):


status,accepted,interview,rejected,submitted,under_review
modality,,,,,
Híbrido,22.58,16.13,22.58,16.13,22.58
Presencial,23.33,16.67,20.00,23.33,16.67
Remoto,18.75,21.88,15.62,25.00,18.75
All,21.51,18.28,19.35,21.51,19.35



Tabla cruzada: Grupo etario vs. Estado de postulación (% por fila, margins=True):


status,accepted,interview,rejected,submitted,under_review
grupo_etario,,,,,
Joven adulto (18-24),22.22,20.83,16.67,22.22,18.06
Menor protegido (16-17),19.05,9.52,28.57,19.05,23.81
All,21.51,18.28,19.35,21.51,19.35


## 4. Tabla dinámica `pivot_table()` por ciudad y modalidad conservando `NaN` (`CA-ISS-11-04` / `CP-051`)

Las combinaciones ciudad × modalidad donde no existen vacantes ni postulaciones se mantienen como `NaN` (sin `fill_value=0`), evitando introducir ceros falsos en aspiraciones salariales o tiempos de respuesta.

In [6]:
pivot_ciudad_modalidad = pd.pivot_table(
    df_u,
    index='ciudad_normalizada',
    columns='modality',
    values=['application_id', 'expected_salary_cop', 'tiempo_respuesta_dias'],
    aggfunc={'application_id': 'count', 'expected_salary_cop': 'median', 'tiempo_respuesta_dias': 'mean'},
)
assert int(pivot_ciudad_modalidad['application_id'].isna().sum().sum()) == 9
assert int(pivot_ciudad_modalidad['expected_salary_cop'].isna().sum().sum()) == 9
assert int(pivot_ciudad_modalidad['tiempo_respuesta_dias'].isna().sum().sum()) == 10
assert int(pivot_ciudad_modalidad['application_id'].sum().sum()) == len(df_u)
pivot_ciudad_modalidad.round(2)

application_id                   expected_salary_cop  \
modality                  Híbrido Presencial Remoto             Híbrido   
ciudad_normalizada                                                        
Barranquilla                  1.0       14.0    NaN           1305884.0   
Bogotá                       16.0        NaN    1.0           2036990.5   
Bucaramanga                  14.0        NaN    NaN           2233809.5   
Cali                          NaN        1.0   15.0                 NaN   
Medellín                      NaN       15.0    NaN                 NaN   
Pereira                       NaN        NaN   16.0                 NaN   

                                         tiempo_respuesta_dias             \
modality           Presencial     Remoto               Híbrido Presencial   
ciudad_normalizada                                                          
Barranquilla        2133032.5        NaN                  7.00        4.0   
Bogotá                    NaN  1387625.0                  4.55        NaN   
Bucaramanga               NaN        NaN                  3.71        NaN   
Cali                1744353.0  2024196.0                   NaN        0.0   
Medellín            1843002.0        NaN                   NaN        4.0   
Pereira                   NaN  1843937.5                   NaN        NaN   

                           
modality           Remoto  
ciudad_normalizada         
Barranquilla          NaN  
Bogotá                NaN  
Bucaramanga           NaN  
Cali                 3.11  
Medellín              NaN  
Pereira              4.56

## 5. Exportación consolidada a `outputs/yeb_kpis_semana6.csv` y validación de sumas e inmutabilidad (`CA-ISS-11-05` / `CP-051B`)

In [7]:
import sys
sys.path.insert(0, str(ANALYTICS_ROOT / 'scripts'))
import importlib
m11 = importlib.import_module('11_kpi_agregaciones_yeb')

kpis_semana6 = m11.construir_kpis_consolidados(df_u)
ruta_kpis = ANALYTICS_ROOT / 'outputs' / 'yeb_kpis_semana6.csv'
kpis_semana6.to_csv(ruta_kpis, index=False, encoding='utf-8', lineterminator='\n')

for dim, g in kpis_semana6.groupby('dimension'):
    assert int(g['n_filas_size'].sum()) == len(df_u), f'Descuadre en {dim}'
    assert int(g['n_respondidas_count'].sum()) == int(df_u['tiempo_respuesta_dias'].count())

huella_despues = hashlib.sha256(RUTA_FUENTE.read_bytes()).hexdigest()
assert HUELLA_ANTES == huella_despues, 'La fuente de Semana 5 fue alterada'
rutas_fichas = m11.verificar_fichas_kpi(ANALYTICS_ROOT)
print(f'Exportado: {ruta_kpis.relative_to(ANALYTICS_ROOT)} ({len(kpis_semana6)} filas en {kpis_semana6["dimension"].nunique()} dimensiones)')
print(f'Inmutabilidad Semana 5 verificada (SHA-256 = {huella_despues[:16]}...) | Fichas verificadas: {len(rutas_fichas)}')
kpis_semana6.loc[kpis_semana6['dimension'].isin(['Global', 'Modalidad', 'Grupo etario'])].round(2)

Exportado: outputs\yeb_kpis_semana6.csv (48 filas en 7 dimensiones)
Inmutabilidad Semana 5 verificada (SHA-256 = 769e94c0a85a651f...) | Fichas verificadas: 2


,dimension,segmento,n_filas_size,n_candidatos_nunique,n_vacantes_nunique,n_respondidas_count,tiempo_resp_media,tiempo_resp_mediana,tiempo_resp_std,n_salarios_count,salario_esp_media,salario_esp_mediana,salario_esp_std,brecha_salarial_media_pct,brecha_salarial_mediana_pct,brecha_salarial_std_pct,n_entrevista_o_contratado,tasa_entrevista_pct,n_contratados,tasa_contratacion_pct
0,Global,Total universo analizable,93,93,18,55,4.00,4.0,2.48,93,2000924.37,1972608.0,386603.74,-2.28,-2.30,9.30,37,39.78,20,21.51
7,Modalidad,Híbrido,31,31,7,19,4.37,4.0,2.31,31,2076415.68,2151971.0,399471.97,-0.44,1.04,10.57,12,38.71,7,22.58
8,Modalidad,Presencial,30,30,7,18,3.78,3.0,2.67,30,1953776.97,1856500.0,397973.40,-2.37,-2.95,8.04,12,40.00,7,23.33
9,Modalidad,Remoto,32,32,7,18,3.83,4.0,2.55,32,1971992.84,1938169.5,363671.68,-3.98,-3.01,9.03,13,40.62,6,18.75
10,Grupo etario,Joven adulto (18-24),72,72,18,43,3.79,3.0,2.52,72,1993811.94,1964328.0,397889.04,-2.37,-2.38,9.41,31,43.06,16,22.22
11,Grupo etario,Menor protegido (16-17),21,21,13,12,4.75,5.5,2.26,21,2025309.81,2051746.0,353129.04,-1.97,-0.64,9.11,6,28.57,4,19.05


## 6. Análisis escrito (150–200 palabras): hallazgos descriptivos, grupos pequeños y decisión no automatizable (`CA-ISS-11-06` / `LEG-001` / `CP-051C`)

El resumen descriptivo sobre el universo analizable de 93 postulaciones válidas al corte del 2026-02-02 muestra una distribución equilibrada por modalidad (Remoto 34,41%, Híbrido 33,33% y Presencial 32,26%) y una tasa global de avance a entrevista o contratación del 39,78%, con un tiempo mediano de respuesta de 4,0 días sobre las 55 postulaciones respondidas y una brecha salarial media de -2,28% frente al punto medio ofertado. Al cruzar ciudad y modalidad aparecen nueve combinaciones vacías (NaN) y tres celdas con un único registro (Barranquilla-Híbrido, Bogotá-Remoto y Cali-Presencial), donde la desviación estándar muestral queda legítimamente en NaN; estos grupos pequeños describen casos puntuales y no permiten comparar ni inferir efectos causales. Asimismo, aunque la cohorte de menores protegidos (16–17 años, n=21) registra un avance descriptivo del 28,57% frente al 43,06% en adultos jóvenes (18–24 años, n=72), ninguna métrica del panel puede automatizar la habilitación ni el descarte de un menor: la verificación de la autorización parental, el registro civil y el permiso de trabajo ante el Ministerio del Trabajo exige revisión humana obligatoria del administrador para proteger los derechos laborales juveniles conforme a la ley.

In [8]:
n_palabras = len(m11.ANALISIS_ESCRITO_KPI.split())
assert 150 <= n_palabras <= 200, f'Fuera de rango (150-200): {n_palabras}'
print(f'Verificación CA-ISS-11-06: {n_palabras} palabras (rango exigido: 150-200). Todo en orden.')

Verificación CA-ISS-11-06: 184 palabras (rango exigido: 150-200). Todo en orden.
